**Imports**

Run once before the other cells.

In [23]:
import json
import sys
import time
from datetime import datetime, timezone
from getpass import getpass
from math import ceil, sqrt
from pathlib import Path
from statistics import NormalDist

import pandas as pd
import requests
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
import gradio as gr
from datasets import load_dataset
import duckdb

### **Week 1 — Option C**

#### **C1: Rules, data and repository**

**Load the 30 items**

Keep official labels hidden and preserve existing manual labels.

In [2]:
print(sys.executable)
path = Path("week1_data.csv")

items = pd.read_csv(path, usecols=["set", "id", "text"])
items = (items.loc[items["set"] == "main", ["id", "text"]]
    .reset_index(drop=True))

assert len(items) == 30
assert items["id"].is_unique
print(f"Ready to label {len(items)} news items.")

labels_path = Path("results/manual_labels.csv")
if labels_path.exists():
    labels = pd.read_csv(labels_path, keep_default_na=False)
    assert labels["id"].tolist() == items["id"].tolist()
else:
    labels = items.copy()
    labels["my_label"] = ""
    labels["my_confidence"] = ""
    labels["my_note"] = ""

print("Labelling table is ready, manually edit csv file with predictions.")
if not labels_path.exists():
    labels_path.parent.mkdir(parents=True, exist_ok=True)
    labels.to_csv(labels_path, index=False)

/Users/ahthini/Desktop/LSA-internship/.venv/bin/python
Ready to label 30 news items.
Labelling table is ready, manually edit csv file with predictions.


**Check manual labels and enter the api key**

Validate all 30 decisions and read the key through hidden input.

In [3]:
labels_path = Path("results/manual_labels.csv")
labels = pd.read_csv(labels_path, keep_default_na=False)

assert len(labels) == 30
assert labels["my_label"].isin(
    ["World", "Sports", "Business", "Sci/Tech"]).all()
assert labels["my_confidence"].isin(["H", "M", "L"]).all()
print("All 30 manual labels are complete.")

api_key = getpass("Enter Groq API key: ").strip()
assert api_key, "No API key entered."
print("Key entered. Ready to test the connection.")

All 30 manual labels are complete.
Key entered. Ready to test the connection.


#### **C2: Structured output**

**Test the api connection**

Classify one made-up headline.

In [4]:
model = "openai/gpt-oss-120b"
api_url = "https://api.groq.com/openai/v1/chat/completions"

#a made-up item for testing the connection.
test_text = "A tennis player won the tournament final in straight sets."
response = requests.post(api_url, headers={
        "Authorization": f"Bearer {api_key}"},
    json={"model": model, "messages": [{
                "role": "user", "content": (
                    "Classify this news item as exactly one of: "
                    "World, Sports, Business, Sci/Tech.\n"
                    "Return only a JSON object with two fields: "
                    "'label' and 'confidence'. "
                    "Confidence must be a number between 0 and 1.\n\n"
                    f"News item: {test_text}")}],
        "temperature": 0,
        "max_completion_tokens": 1024}, timeout=60)

print("HTTP status:", response.status_code)

if response.ok:
    reply = response.json()["choices"][0]["message"]["content"]
    print("Model reply:")
    print(reply)
else:
    print(response.text.replace(api_key, "[REDACTED]"))

HTTP status: 200
Model reply:
{
  "label": "Sports",
  "confidence": 0.99
}


**Validate the model reply**

Check JSON, category and confidence. Record invalid replies.

In [5]:
categories = {"World", "Sports", "Business", "Sci/Tech"}
def validate_reply(reply):
    try:
        result = json.loads(reply)

        if not isinstance(result, dict):
            raise ValueError("Reply must be a JSON object.")

        if result.get("label") not in categories:
            raise ValueError("Missing or invalid category.")

        confidence = result.get("confidence")

        if type(confidence) not in (int, float):
            raise ValueError("Confidence must be a number.")

        if not 0 <= confidence <= 1:
            raise ValueError("Confidence must be between 0 and 1.")

        return {"ai_label": result["label"],
            "ai_confidence": confidence,
            "valid": True,
            "error": "",
            "raw_reply": reply}

    except (ValueError, TypeError) as error:
        return {"ai_label": None,
            "ai_confidence": None,
            "valid": False,
            "error": str(error),
            "raw_reply": reply}

validated = validate_reply(reply)
validated

{'ai_label': 'Sports',
 'ai_confidence': 0.99,
 'valid': True,
 'error': '',
 'raw_reply': '{\n  "label": "Sports",\n  "confidence": 0.99\n}'}

**Run both prompts**

Classify each item with both prompts and save every response.

In [6]:
#both supplied prompts use the same json output instructions for option c.
prompt_1 = """
You are classifying news items into exactly one of four categories.

World: international news, politics, conflict, disasters, national affairs.
Sports: matches, results, athletes, teams, transfers, tournaments.
Business: companies, markets, earnings, the economy, trade, jobs, deals.
Sci/Tech: science, research, technology, software, space, health research.

Boundary rules, apply them in this order:
1. A foreign company's earnings or a currency move is Business, not World.
2. A broadcasting rights deal or a club's finances is Business, not Sports.
3. A product's technical capability is Sci/Tech. Its commercial impact is Business.
4. If two categories still fit, choose the one the item is mostly about.

Return only a JSON object with:
- "label": exactly World, Sports, Business, or Sci/Tech
- "confidence": a number between 0 and 1

News item:
{text}
"""

prompt_2 = """
What category is this news item? Pick one.
World, Sports, Business, Sci/Tech.

Return only a JSON object with:
- "label": exactly World, Sports, Business, or Sci/Tech
- "confidence": a number between 0 and 1

News item:
{text}
"""

prompts = {"specific": prompt_1, "vague": prompt_2}


def classify_one(text, template):
    #substitute news item into the prompt
    prompt = template.format(text=text)

    for attempt in range(4):
        response = requests.post(
            api_url,
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": model,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0,
                "max_completion_tokens": 1024}, timeout=60)

        #429 means rate limit hit
        if response.status_code == 429 and attempt < 3:
            try:
                retry_after = float(response.headers.get("retry-after", "0"))
            except ValueError:
                retry_after = 0

            wait = max(retry_after, 4 * (2 ** attempt))
            print(f"Rate limited. Waiting {wait:.0f} seconds.")
            time.sleep(wait)
            continue

        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]

#read only the fields the ai needs.
news = pd.read_csv("results/manual_labels.csv",
    usecols=["id", "text"], keep_default_na=False)

assert len(news) == 30
assert news["id"].is_unique
assert news["text"].str.strip().ne("").all()

output_path = Path("results/ai_labels.csv")

#reuse saved responses instead of repeating requests or overwriting results
if output_path.exists():
    ai_results = pd.read_csv(output_path, keep_default_na=False)
    expected_pairs = {(item_id, name) for item_id in news["id"] for name in prompts}
    saved_pairs = set(zip(ai_results["id"], ai_results["prompt"]))
    assert not ai_results.duplicated(["id", "prompt"]).any(), "saved results contain duplicate items"
    assert saved_pairs == expected_pairs, "saved results are incomplete or differ from this dataset. review before continuing"
    records = ai_results.to_dict("records")
    print(f"loaded {len(records)} saved results from {output_path}. api loop skipped.")
else:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    records = []

    for prompt_name, template in prompts.items():
        for position, row in enumerate(news.itertuples(index=False), start=1):
            try:
                raw_reply = classify_one(row.text, template)
                result = validate_reply(raw_reply)
                status = "ok" if result["valid"] else "invalid_output"

            except requests.RequestException as error:
                #connection failures are different from malformed ai replies.
                result = {"ai_label": None,
                    "ai_confidence": None,
                    "valid": False,
                    "error": str(error).replace(api_key, "[REDACTED]"),
                    "raw_reply": ""}
                status = "api_error"

            except (ValueError, KeyError, IndexError, TypeError) as error:
                result = {"ai_label": None,
                    "ai_confidence": None,
                    "valid": False,
                    "error": f"Unexpected API response: {error}",
                    "raw_reply": ""}
                status = "response_error"

            records.append({"id": row.id,
                "prompt": prompt_name,
                "model": model,
                "prompt_template": template,
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "status": status,
                **result})

            #save after every item so completed work survives interruption.
            pd.DataFrame(records).to_csv(output_path, index=False)

            print(f"{prompt_name}: {position}/30 — {status}")

            #pause between requests as instructed in the assignment.
            time.sleep(4)

    print(f"Saved {len(records)} results to {output_path}")


loaded 60 saved results from results/ai_labels.csv. api loop skipped.


**Check saved results**

Count valid replies, failures and the invalid-output rate.

In [7]:
ai_results = pd.read_csv(
    "results/ai_labels.csv",
    keep_default_na=False)

summary = (ai_results.groupby("prompt")
    .agg(attempted=("id", "size"),
        valid_outputs=("status", lambda s: (s == "ok").sum()),
        invalid_outputs=(
            "status", lambda s: (s == "invalid_output").sum()),
        api_errors=("status", lambda s: (s == "api_error").sum()),
        response_errors=(
            "status", lambda s: (s == "response_error").sum())))

#parse/validation failures among replies received from the model.
received = summary["valid_outputs"] + summary["invalid_outputs"]
summary["invalid_output_rate"] = (
    summary["invalid_outputs"] / received.replace(0, float("nan")))

summary

,attempted,valid_outputs,invalid_outputs,api_errors,response_errors,invalid_output_rate
prompt,,,,,,
specific,30,30,0,0,0,0.0
vague,30,30,0,0,0,0.0


#### **C3: Measure agreement**

**Reveal official labels and compare**

Match by ID and calculate pairwise agreement and kappa.

In [8]:
#load saved decisions without rerunning classification
manual = pd.read_csv("results/manual_labels.csv", keep_default_na=False)
ai = pd.read_csv("results/ai_labels.csv", keep_default_na=False)
assert len(manual) == 30 and manual["id"].is_unique
assert len(ai) == 60 and not ai.duplicated(["id", "prompt"]).any()
assert ai["status"].eq("ok").all(), "review failed responses before scoring"
assert set(ai["prompt"]) == {"specific", "vague"}
for name in ["specific", "vague"]:
    assert set(ai.loc[ai["prompt"] == name, "id"]) == set(manual["id"])

#reveal the official labels only now
truth = pd.read_csv("week1_data.csv", usecols=["set", "id", "official_label"])
truth = truth.loc[truth["set"] == "main", ["id", "official_label"]]
assert len(truth) == 30 and truth["id"].is_unique
assert set(truth["id"]) == set(manual["id"])

wide = ai.pivot(index="id", columns="prompt", values="ai_label")
wide = wide.rename(columns={"specific": "ai_specific", "vague": "ai_vague"}).reset_index()
comparison = manual.merge(wide, on="id", validate="one_to_one")
comparison = comparison.merge(truth, on="id", validate="one_to_one")
assert len(comparison) == 30
comparison.to_csv("results/label_comparison.csv", index=False)
display(comparison)

#the human-versus-official pair is shared by both prompt experiments
pairs = [("you vs official", "my_label", "official_label"),
         ("you vs specific ai", "my_label", "ai_specific"),
         ("specific ai vs official", "ai_specific", "official_label"),
         ("you vs vague ai", "my_label", "ai_vague"),
         ("vague ai vs official", "ai_vague", "official_label")]
metrics = []
for name, left, right in pairs:
    matches = comparison[left].eq(comparison[right])
    metrics.append({"comparison": name, "matches": int(matches.sum()),
                    "items": len(comparison), "agreement": matches.mean(),
                    "kappa": cohen_kappa_score(comparison[left], comparison[right])})

agreement_results = pd.DataFrame(metrics)
agreement_results.to_csv("results/agreement_scores.csv", index=False)
display(agreement_results.round(3))

,id,text,my_label,my_confidence,my_note,ai_specific,ai_vague,official_label
0,1,Gerrard only a 50-50 chance VIENNA - England m...,Sports,H,"Mentions of World Cup, midfielder, training",Sports,Sports,Sports
1,2,A Better Solution for Israel The hysterical to...,World,M,"Mentions of Israel, ususally political, and ta...",World,World,World
2,3,"Olympic Daily Preview - Thursday, August 19 At...",Sports,H,Mentions of Olympics and sports,Sports,Sports,Sports
3,4,"US men #39;s basketball routs Angola Athens, G...",Sports,H,"Mentions of basketball, scores, and Sports Net...",Sports,Sports,Sports
4,5,"Italy: Sicilian derby ends in draw PALERMO, No...",Sports,H,Mentions match and draw,Sports,Sports,Sports
5,6,5 of arthritis patients in Singapore take Bext...,Sci/Tech,H,"Mentions arthritis (medical condition), prescr...",Sci/Tech,Sci/Tech,Business
6,7,Prince Charles chastised for quot;old fashione...,World,H,Mentions Prince Charles (royalty) and he is th...,World,World,World
7,8,Dollar Idles vs. Euro Before U.S. Data LONDON ...,Business,H,"Mentions of investors, dollar (currency)",Business,Business,Business
8,9,Infineon to Pay a Fine in the Fixing Of Chip P...,Sci/Tech,L,"Mentions of tech, global cartel (world confusi...",Business,Business,Business
9,10,Images: Nintendo grows up--a little The Ninten...,Sci/Tech,H,Mentions of tech (Nintendo),Sci/Tech,Sci/Tech,Sci/Tech


,comparison,matches,items,agreement,kappa
0,you vs official,22,30,0.733,0.643
1,you vs specific ai,25,30,0.833,0.779
2,specific ai vs official,24,30,0.800,0.733
3,you vs vague ai,26,30,0.867,0.823
4,vague ai vs official,24,30,0.800,0.733


**Confusion matrices and category performance**

Compare predictions against official labels. Rows are official categories and
columns are predicted categories.

In [9]:
#reload saved results
comparison = pd.read_csv("results/label_comparison.csv", keep_default_na=False)
category_order = ["World", "Sports", "Business", "Sci/Tech"]
predictions = {"human": "my_label", "specific": "ai_specific", "vague": "ai_vague"}

for name, column in predictions.items():
    print(f"\n{name}")
    matrix = confusion_matrix(comparison["official_label"],
                              comparison[column], labels=category_order)
    matrix = pd.DataFrame(matrix, index=category_order, columns=category_order)
    matrix.index.name = "official"
    matrix.columns.name = "predicted"
    display(matrix)
    matrix.to_csv(f"results/confusion_matrix_{name}.csv")

    #calculate performance for each category
    report = classification_report(comparison["official_label"],
                                   comparison[column], labels=category_order,
                                   output_dict=True, zero_division=0)
    accuracy = report.pop("accuracy")
    print(f"accuracy: {accuracy:.1%}")
    report = pd.DataFrame(report).T
    display(report.round(3))
    report.to_csv(f"results/category_metrics_{name}.csv")


human


predicted,World,Sports,Business,Sci/Tech
official,,,,
World,7,0,1,0
Sports,0,8,0,0
Business,0,1,2,4
Sci/Tech,1,0,1,5


accuracy: 73.3%


,precision,recall,f1-score,support
World,0.875,0.875,0.875,8.0
Sports,0.889,1.000,0.941,8.0
Business,0.500,0.286,0.364,7.0
Sci/Tech,0.556,0.714,0.625,7.0
macro avg,0.705,0.719,0.701,30.0
weighted avg,0.717,0.733,0.715,30.0



specific


...

,precision,recall,f1-score,support
World,0.875,0.875,0.875,8.0
Sports,1.000,1.000,1.000,8.0
Business,0.625,0.714,0.667,7.0
Sci/Tech,0.667,0.571,0.615,7.0
macro avg,0.792,0.790,0.789,30.0
weighted avg,0.801,0.800,0.799,30.0



vague


predicted,World,Sports,Business,Sci/Tech
official,,,,
World,7,0,1,0
Sports,0,8,0,0
Business,0,1,5,1
Sci/Tech,1,0,2,4


accuracy: 80.0%


,precision,recall,f1-score,support
World,0.875,0.875,0.875,8.0
Sports,0.889,1.000,0.941,8.0
Business,0.625,0.714,0.667,7.0
Sci/Tech,0.800,0.571,0.667,7.0
macro avg,0.797,0.790,0.787,30.0
weighted avg,0.803,0.800,0.795,30.0


**Demonstrate data leakage**

Compare held-out accuracy with accuracy after including test items in training.

In [10]:
#use the supplied 200-item sample for this experiment
data = pd.read_csv("week1_data.csv")
pool = data.loc[data["set"] == "sorted_sample",
                ["id", "text", "official_label"]].reset_index(drop=True)

assert len(pool) == 200

#hold out 30% while preserving approximately the same category proportions
train_text, test_text, train_labels, test_labels = train_test_split(
    pool["text"], pool["official_label"], test_size=0.3,
    random_state=0, stratify=pool["official_label"])

#honest: learn vocabulary and classifier from training items only
honest = make_pipeline(TfidfVectorizer(),
                       LogisticRegression(max_iter=1000, random_state=0))
honest.fit(train_text, train_labels)
honest_predictions = honest.predict(test_text)
honest_accuracy = accuracy_score(test_labels, honest_predictions)

#deliberately leaky: training includes the test items and their answers
leaky = make_pipeline(TfidfVectorizer(),
                      LogisticRegression(max_iter=1000, random_state=0))
leaky.fit(pool["text"], pool["official_label"])
leaky_predictions = leaky.predict(test_text)
leaky_accuracy = accuracy_score(test_labels, leaky_predictions)

#compare both models on exactly the same test items
leakage_results = pd.DataFrame([
    {"evaluation": "honest", "training_items": len(train_text),
     "test_items": len(test_text), "accuracy": honest_accuracy},
    {"evaluation": "leaky", "training_items": len(pool),
     "test_items": len(test_text), "accuracy": leaky_accuracy}])

leakage_results.to_csv("results/leakage_comparison.csv", index=False)
display(leakage_results)

gap = 100 * (leaky_accuracy - honest_accuracy)
print(f"accuracy difference: {gap:.1f} percentage points")

,evaluation,training_items,test_items,accuracy
0,honest,140,60,0.616667
1,leaky,200,60,1.000000


accuracy difference: 38.3 percentage points


**Evaluate an imbalanced dataset**

Predict World for every item and compare accuracy with kappa.

In [11]:
#load the separate imbalanced sample
data = pd.read_csv("week1_data.csv")
imbalanced = data.loc[data["set"] == "imbalanced"].reset_index(drop=True)
assert len(imbalanced) == 30

#predict the same category without using the text
always_world = ["World"] * len(imbalanced)
truth = imbalanced["official_label"]

class_counts = truth.value_counts().rename_axis("category").reset_index(name="items")
display(class_counts)

imbalance_results = pd.DataFrame([
    {"method": "always world", "items": len(imbalanced),
     "accuracy": accuracy_score(truth, always_world),
     "kappa": cohen_kappa_score(truth, always_world)}])

display(imbalance_results)
imbalance_results.to_csv("results/imbalance_scores.csv", index=False)
class_counts.to_csv("results/imbalanced_class_counts.csv", index=False)

,category,items
0,World,24
1,Sci/Tech,2
2,Business,2
3,Sports,2


,method,items,accuracy,kappa
0,always world,30,0.8,0.0


#### **C4: Quantify uncertainty**

**Calculate confidence intervals**

Use 95% Wilson intervals to show uncertainty in agreement measured on 30 items.

In [12]:
def wilson_interval(matches, total, confidence=0.95):
    #calculate uncertainty around a proportion
    assert total > 0 and 0 <= matches <= total
    z = NormalDist().inv_cdf(0.5 + confidence / 2)
    proportion = matches / total
    denominator = 1 + z**2 / total
    centre = (proportion + z**2 / (2 * total)) / denominator
    margin = z * sqrt(proportion * (1 - proportion) / total
                      + z**2 / (4 * total**2)) / denominator
    return centre - margin, centre + margin

#use actual saved match counts
agreement_scores = pd.read_csv("results/agreement_scores.csv")
intervals = []

for row in agreement_scores.itertuples(index=False):
    lower, upper = wilson_interval(row.matches, row.items)
    intervals.append({"comparison": row.comparison,
                      "matches": row.matches, "items": row.items,
                      "agreement": row.agreement,
                      "lower_95": lower, "upper_95": upper})

confidence_intervals = pd.DataFrame(intervals)
confidence_intervals.to_csv("results/agreement_intervals.csv", index=False)

#format a display copy and keep saved results numeric
interval_display = confidence_intervals.copy()
for column in ["agreement", "lower_95", "upper_95"]:
    interval_display[column] = interval_display[column].map("{:.1%}".format)
display(interval_display)

#choose the precision before collecting more evaluation data
confidence = 0.95
target_margin = 0.05
z = NormalDist().inv_cdf(0.5 + confidence / 2)

#0.5 gives the conservative planning estimate for a proportion
planning_proportion = 0.5
required_items = ceil(z**2 * planning_proportion
                      * (1 - planning_proportion) / target_margin**2)

sample_plan = pd.DataFrame([
    {"confidence": confidence, "target_margin": target_margin,
     "planning_proportion": planning_proportion,
     "required_items": required_items}])

display(sample_plan)
sample_plan.to_csv("results/evaluation_sample_plan.csv", index=False)

print(f"plan for approx {required_items} independent, representative items")
print(f"one changed result in the current sample moves accuracy by {100 / 30:.1f} percentage points")

,comparison,matches,items,agreement,lower_95,upper_95
0,you vs official,22,30,73.3%,55.6%,85.8%
1,you vs specific ai,25,30,83.3%,66.4%,92.7%
2,specific ai vs official,24,30,80.0%,62.7%,90.5%
3,you vs vague ai,26,30,86.7%,70.3%,94.7%
4,vague ai vs official,24,30,80.0%,62.7%,90.5%


,confidence,target_margin,planning_proportion,required_items
0,0.95,0.05,0.5,385


plan for approx 385 independent, representative items
one changed result in the current sample moves accuracy by 3.3 percentage points


#### **C5: Try to break it**

**Test difficult inputs**

Record responses to empty text, noise, another language and an ambiguous headline.

In [13]:
#use fixed test inputs so the experiment can be repeated
test_cases = {"empty": "",
    "noise": " ".join(["zxqv", "plmok", "qwrty"] * 1000),
    "other_language": "El equipo ganó la final del torneo de fútbol.",
    "ambiguous": "A football club signs a record broadcasting deal to increase its revenue."}

robustness_path = Path("results/robustness_before.csv")

#reuse saved results rather than repeat api calls
if robustness_path.exists():
    robustness_before = pd.read_csv(robustness_path, keep_default_na=False)
else:
    records = []

    for name, text in test_cases.items():
        try:
            raw_reply = classify_one(text, prompt_1)
            result = validate_reply(raw_reply)
            status = "ok" if result["valid"] else "invalid_output"

        except requests.RequestException as error:
            result = {"ai_label": None, "ai_confidence": None,
                      "valid": False,
                      "error": str(error).replace(api_key, "[REDACTED]"),
                      "raw_reply": ""}
            status = "api_error"

        except (ValueError, KeyError, IndexError, TypeError) as error:
            result = {"ai_label": None, "ai_confidence": None,
                      "valid": False, "error": str(error),
                      "raw_reply": ""}
            status = "response_error"

        records.append({"case": name, "input_text": text,
                        "word_count": len(text.split()),
                        "model": model, "status": status, **result})
        print(f"{name}: {status}")
        time.sleep(4)

    robustness_before = pd.DataFrame(records)
    robustness_before.to_csv(robustness_path, index=False)

display(robustness_before[["case", "word_count", "ai_label",
                          "ai_confidence", "status", "error"]])

,case,word_count,ai_label,ai_confidence,status,error
0,empty,0,World,0.0,ok,
1,noise,3000,,,api_error,413 Client Error: Payload Too Large for url: h...
2,other_language,9,Sports,0.99,ok,
3,ambiguous,12,Business,0.97,ok,


**Compare behaviour before and after**

Check that invalid inputs are rejected before an API request.

In [14]:
def check_input(text):
    #require text with content rather than whitespace alone
    if not isinstance(text, str):
        raise ValueError("input must be text")
    if not text.strip():
        raise ValueError("input must not be empty")
    if len(text.split()) > 2000:
        raise ValueError("input must contain no more than 2,000 words")
    return text.strip()

def classify_checked(text, template):
    #validate before sending anything to the model
    checked_text = check_input(text)
    return classify_one(checked_text, template)

#both inputs should now be rejected before reaching the model
after_records = []

for name in ["empty", "noise"]:
    before = robustness_before.loc[robustness_before["case"] == name].iloc[0]

    try:
        check_input(test_cases[name])
        after_status = "accepted"
        after_message = ""
    except ValueError as error:
        after_status = "rejected"
        after_message = str(error)

    after_records.append({"case": name,
                          "before_status": before["status"],
                          "before_label": before["ai_label"],
                          "before_confidence": before["ai_confidence"],
                          "after_status": after_status,
                          "after_message": after_message})

robustness_after = pd.DataFrame(after_records)
robustness_after.to_csv("results/robustness_after.csv", index=False)
display(robustness_after)

assert robustness_after["after_status"].eq("rejected").all()


,case,before_status,before_label,before_confidence,after_status,after_message
0,empty,ok,World,0.0,rejected,input must not be empty
1,noise,api_error,,,rejected,"input must contain no more than 2,000 words"


#### **C6: Interface and endpoint**

**Build the interface**

Install Gradio once, then run the Imports cell. Reuse input checks and validate every model reply.

In [15]:
def classify_for_app(text):
    #reject invalid input before calling the model
    try:
        checked_text = check_input(text)
    except ValueError as error:
        raise gr.Error(str(error)) from None

    if not globals().get("api_key"):
        raise gr.Error("enter your api key in the notebook before using the app")

    try:
        raw_reply = classify_checked(checked_text, prompt_1)
    except requests.HTTPError as error:
        status = error.response.status_code if error.response is not None else "unknown"
        raise gr.Error(f"model service returned HTTP {status}. check the notebook connection test") from None
    except requests.RequestException:
        raise gr.Error("could not reach the model service. try again shortly") from None
    except (ValueError, KeyError, IndexError, TypeError):
        raise gr.Error("the model service returned an unexpected response") from None

    result = validate_reply(raw_reply)
    if not result["valid"]:
        raise gr.Error(f"invalid model output: {result['error']}")
    return result["ai_label"], result["ai_confidence"]

#close a previous interface before rebuilding it
if "demo" in globals():
    demo.close()

demo = gr.Interface(fn=classify_for_app,
    inputs=gr.Textbox(label="News item", lines=6,
                     placeholder="Paste a news headline or short article"),
    outputs=[gr.Textbox(label="Category"), gr.Number(label="Model-reported confidence")],
    title="Week 1 news classifier",
    description="Classify news as World, Sports, Business or Sci/Tech. Maximum 2,000 words. Confidence is the model's estimate, not verified accuracy.",
    examples=[["A tennis player won the tournament final in straight sets."]],
    cache_examples=False, api_name="classify", concurrency_limit=1)


**Launch and test**

Open the public link on your phone and try a headline and empty input. Open Use via API and save a screenshot. Keep the notebook running while testing.

In [16]:
#launch creates a public link while this notebook is running
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://e1f21ccb8ca7a8fe1a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#### **C7: Evaluate project labels**

**Prepare 30 BESSTIE items**

Label sentiment and sarcasm independently. Reference labels are excluded from the editing file.

In [19]:
#use validation data and a fixed seed rather than choosing favourable examples
besstie = load_dataset("surrey-nlp/BESSTIE-CW-26", split="validation")
besstie_path = Path("results/besstie_manual_labels.csv")
besstie_meta_path = Path("results/besstie_sample.json")

#preserve existing decisions on reruns
if besstie_path.exists():
    besstie_manual = pd.read_csv(besstie_path, keep_default_na=False)
    assert besstie_meta_path.exists(), "sample metadata is missing"
    sample_meta = json.loads(besstie_meta_path.read_text())
    assert sample_meta["fingerprint"] == besstie._fingerprint, "dataset changed since sampling"
    print("loaded existing labelling file without overwriting it")
else:
    assert not besstie_meta_path.exists(), "sample metadata already exists. check the previous sample"
    sample_ids = pd.Series(range(len(besstie))).sample(n=30, random_state=42).tolist()
    besstie_manual = pd.DataFrame({"id": sample_ids,
                                  "text": [besstie[int(i)]["text"] for i in sample_ids]})
    for column in ["my_sentiment", "sentiment_confidence", "my_sarcasm", "sarcasm_confidence", "my_note"]:
        besstie_manual[column] = ""
    sample_meta = {"dataset": "surrey-nlp/BESSTIE-CW-26", "split": "validation",
                   "seed": 42, "fingerprint": besstie._fingerprint, "ids": sample_ids}
    besstie_path.parent.mkdir(parents=True, exist_ok=True)
    besstie_manual.to_csv(besstie_path, index=False)
    besstie_meta_path.write_text(json.dumps(sample_meta, indent=2))

print(f"label these 30 texts in: {besstie_path.resolve()}")
print("sentiment: 0 = negative, 1 = positive")
print("sarcasm: 0 = non-sarcastic, 1 = sarcastic")
print("confidence: H, M or L for each task")


Generating test split: 100%|██████████| 2183/2183 [00:00<00:00, 138417.29 examples/s]

label these 30 texts in: /Users/ahthini/Desktop/LSA-internship/Week 1/results/besstie_manual_labels.csv
sentiment: 0 = negative, 1 = positive
sarcasm: 0 = non-sarcastic, 1 = sarcastic
confidence: H, M or L for each task


**Compare your labels with the dataset**

Run only after saving all 30 decisions for both tasks. Calculate agreement and kappa separately.

In [20]:
#reload your saved decisions before allowing reference labels to be revealed
besstie_manual = pd.read_csv(besstie_path, keep_default_na=False)
assert len(besstie_manual) == 30 and besstie_manual["id"].is_unique
assert besstie_manual["id"].tolist() == sample_meta["ids"], "sample ids or order changed"
for column in ["my_sentiment", "my_sarcasm"]:
    values = pd.to_numeric(besstie_manual[column], errors="coerce")
    assert values.isin([0, 1]).all(), f"complete all 30 values in {column} before revealing labels"
    besstie_manual[column] = values.astype(int)
for column in ["sentiment_confidence", "sarcasm_confidence"]:
    assert besstie_manual[column].isin(["H", "M", "L"]).all(), f"complete {column}"

#match references by their original validation-row ids
reference_rows = [besstie[int(i)] for i in besstie_manual["id"]]
assert besstie_manual["text"].tolist() == [row["text"] for row in reference_rows]
besstie_comparison = besstie_manual.copy()
besstie_comparison["official_sentiment"] = [int(row["Sentiment"]) for row in reference_rows]
besstie_comparison["official_sarcasm"] = [int(row["Sarcasm"]) for row in reference_rows]
besstie_comparison.to_csv("results/besstie_comparison.csv", index=False)

#evaluate the two tasks separately
scores = []
for task in ["sentiment", "sarcasm"]:
    mine = besstie_comparison[f"my_{task}"]
    official = besstie_comparison[f"official_{task}"]
    scores.append({"task": task, "matches": int(mine.eq(official).sum()),
                   "items": len(mine), "agreement": accuracy_score(official, mine),
                   "kappa": cohen_kappa_score(official, mine)})

besstie_scores = pd.DataFrame(scores)
besstie_scores.to_csv("results/besstie_scores.csv", index=False)
display(besstie_scores.round(3))
display(besstie_comparison)

,task,matches,items,agreement,kappa
0,sentiment,25,30,0.833,0.667
1,sarcasm,27,30,0.900,0.609


,id,text,my_sentiment,sentiment_confidence,my_sarcasm,sarcasm_confidence,my_note,official_sentiment,official_sarcasm
0,46,My in the lunch line buying it for 1.20 (so ex...,0,M,0,M,,0,1
1,9,"Very interesting Museum, quite small but very ...",1,H,0,H,,1,0
2,206,Waited 20 minutes for my drink. Lady was too i...,0,H,0,M,,0,0
3,139,70% tax is wild,0,M,0,M,,0,0
4,60,Went in for breakfast on saturday during our v...,0,L,0,H,Very mixed review,1,0
5,33,Oh...she is going to get roasted so much. Aiyo.,0,M,0,M,Aiyo is a south asian saying like oh my god (n...,1,0
6,297,"A bit overpriced, but still tasty. We were a b...",0,M,0,H,,1,0
7,186,I was so excited to finally visit my first Jol...,0,H,0,H,,0,0
8,296,Very calm place to dine in. Moreover this is t...,1,H,0,H,,0,0
9,277,"This guy is an imbecile, Indian vyomnauts are ...",0,H,1,M,,0,0


In [21]:
#show disagreements without changing the original decisions
besstie_comparison = pd.read_csv("results/besstie_comparison.csv",
                                keep_default_na=False)

for task in ["sentiment", "sarcasm"]:
    different = besstie_comparison[
        besstie_comparison[f"my_{task}"] != besstie_comparison[f"official_{task}"]]

    print(f"\n{task.upper()} DISAGREEMENTS")

    for row in different.to_dict("records"):
        print(f"\nid: {row['id']}")
        print(row["text"])
        print(f"my label: {row[f'my_{task}']}")
        print(f"reference label: {row[f'official_{task}']}")
        print(f"confidence: {row[f'{task}_confidence']}")
        print(f"my note: {row['my_note']}")
        print("-" * 60)


SENTIMENT DISAGREEMENTS

id: 60
Went in for breakfast on saturday during our valentines weekend. It was busy, luckily we got seated within minutes because the line outside was crazy after. I could n't believe that people were waiting to get inside. I had the eggs Benedict and my partner had the full English breakfast. I would n't say it amazing as I had better but it was really nice. The service was good, staff were polite, professional and attentive. The pancakes were popular as I could see several others order it. Overall, great! Would I go again ? Not really .
my label: 0
reference label: 1
confidence: L
my note: Very mixed review
------------------------------------------------------------

id: 33
Oh...she is going to get roasted so much. Aiyo.
my label: 0
reference label: 1
confidence: M
my note: Aiyo is a south asian saying like oh my god (negatively)
------------------------------------------------------------

id: 297
A bit overpriced, but still tasty. We were a bit cautious c

**Explore keyword retrieval**

Find news items containing the word world and inspect their categories.

In [22]:
#search for a word rather than its meaning
comparison = pd.read_csv("results/label_comparison.csv", keep_default_na=False)
world_matches = comparison.loc[
    comparison["text"].str.contains(r"\bworld\b", case=False, na=False),
    ["id", "text", "official_label"]]

for row in world_matches.itertuples(index=False):
    print(f"\nid: {row.id} | official category: {row.official_label}")
    print(row.text)


id: 1 | official category: Sports
Gerrard only a 50-50 chance VIENNA - England midfielder Steven Gerrard is a major doubt to face Austria in today #39;s opening 2006 World Cup qualifier. Gerrard has failed to shake off a groin injury suffered in training on Thursday and

id: 16 | official category: World
Cocoa prices down but Ivorian exports still blocked World cocoa prices were down on Tuesday but off the day #39;s lows as exports remained on hold after mob violence and military clashes in Ivory Coast, the key global supplier, traders said.

id: 17 | official category: World
China 'Deeply Concerned' at Japan's Defense Move SHANGHAI (Reuters) - China voiced deep concern on Saturday over Japan's sweeping overhaul of defense policy that eased a decades-old ban on arms exports and suggested a shift from a defensive posture in place since its World War II defeat.

id: 20 | official category: World
World Population to Rise to 8.9 Billion Although world families are getting smaller in many 

In [24]:
#perform the same whole-word search using sql
sql_matches = duckdb.query("""
    SELECT id, text, official_label
    FROM comparison
    WHERE regexp_matches(lower(text), '\\bworld\\b')
""").df()

assert set(sql_matches["id"]) == set(world_matches["id"])
display(sql_matches)

,id,text,official_label
0,1,Gerrard only a 50-50 chance VIENNA - England m...,Sports
1,16,Cocoa prices down but Ivorian exports still bl...,World
2,17,China 'Deeply Concerned' at Japan's Defense Mo...,World
3,20,World Population to Rise to 8.9 Billion Althou...,World
4,29,Siemens Says Cellphone Flaw May Hurt Users and...,Business


In [26]:
#inspect each prompt separately without changing original labels
for name, column in {"specific": "ai_specific", "vague": "ai_vague"}.items():
    shared = comparison.loc[
        comparison["my_label"].eq(comparison[column])
        & comparison["my_label"].ne(comparison["official_label"])]

    print(f"\n{name}: {len(shared)} shared disagreements")

    for row in shared.to_dict("records"):
        print(f"\nid: {row['id']}")
        print(row["text"])
        print(f"my label and ai: {row['my_label']}")
        print(f"official: {row['official_label']}")
        print(f"my note: {row['my_note']}")

#check whether lower confidence corresponds to more disagreements
confidence_check = comparison.assign(
    disagrees=comparison["my_label"].ne(comparison["official_label"]))

confidence_summary = confidence_check.groupby("my_confidence").agg(
    items=("id", "size"), disagreements=("disagrees", "sum"),
    disagreement_rate=("disagrees", "mean")).reindex(["H", "M", "L"])

display(confidence_summary.round(3))
confidence_summary.to_csv("results/manual_confidence_summary.csv")


specific: 4 shared disagreements

id: 6
5 of arthritis patients in Singapore take Bextra or Celebrex &lt;b&gt;...&lt;/b&gt; SINGAPORE : Doctors in the United States have warned that painkillers Bextra and Celebrex may be linked to major cardiovascular problems and should not be prescribed.
my label and ai: Sci/Tech
official: Business
my note: Mentions arthritis (medical condition), prescribed, cardiovascular, and medicine names

id: 16
Cocoa prices down but Ivorian exports still blocked World cocoa prices were down on Tuesday but off the day #39;s lows as exports remained on hold after mob violence and military clashes in Ivory Coast, the key global supplier, traders said.
my label and ai: Business
official: World
my note: Mentions of traders, a quote by them

id: 18
Shock Jock Group Boosts Satellite Radio Profile When radio shock jocks Opie and Anthony considered their next career move after two firings in four years, the twisted twosome was ready to feign rehabilitation. Or at least

,items,disagreements,disagreement_rate
my_confidence,,,
H,21,3,0.143
M,7,3,0.429
L,2,2,1.000
